# Prepare the LAION-Art AMP attack set

Assign each clean image its highest-scoring caption concept using AMP's concept-selection procedure, then reproducibly sample source-target pairs. This notebook only prepares clean inputs; it does not generate adversarial images.

In [ ]:
from pathlib import Path
import json
import random
import shutil

import nltk
import numpy as np
import open_clip
import pandas as pd
import spacy
import torch
from PIL import Image
from nltk.stem import WordNetLemmatizer
from tqdm.auto import tqdm

DATA_DIR = Path("dataset/laion_art")
CLEAN_DIR = DATA_DIR / "clean"
METADATA_PATH = DATA_DIR / "metadata.csv"
ASSIGNMENTS_PATH = DATA_DIR / "concept_assignments.csv"
PAIRS_PATH = DATA_DIR / "concept_pairs.csv"
ATTACK_DIR = DATA_DIR / "attack_set"

NUM_CONCEPT_PAIRS = 10
IMAGES_PER_PAIR = 2
RANDOM_SEED = 42
IMAGE_BATCH_SIZE = 32
TEXT_BATCH_SIZE = 256
REBUILD_ASSIGNMENTS = False

MODEL_NAME = "EVA02-E-14-plus"
PRETRAINED = "laion2b_s9b_b144k"
device = "cuda" if torch.cuda.is_available() else "cpu"
if device != "cuda":
    raise RuntimeError("A CUDA GPU is required for efficient concept scoring.")

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
print(f"Using {device}: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load captions and match them to clean image stems.
metadata = pd.read_csv(METADATA_PATH)
images = sorted(CLEAN_DIR.glob("*.png"))
if not images:
    raise FileNotFoundError(f"No PNG images found in {CLEAN_DIR}")

metadata = metadata.loc[metadata["status"].eq("complete") & metadata["final_path"].notna()].copy()
metadata["image_id"] = metadata["final_path"].map(lambda path: Path(path).stem)
if metadata["image_id"].duplicated().any():
    raise ValueError(f"Duplicate image IDs in {METADATA_PATH}")
caption_by_id = metadata.set_index("image_id")["TEXT"]
records = pd.DataFrame({
    "image_path": [path.as_posix() for path in images],
    "image_id": [path.stem for path in images],
})
records["caption"] = records["image_id"].map(caption_by_id)
if records["caption"].isna().any():
    missing = records.loc[records["caption"].isna(), "image_id"].tolist()[:10]
    raise ValueError(f"Missing LAION captions for image IDs: {missing}")
records.head()

In [ ]:
# AMP extracts NOUN/PROPN tokens, lowercases them, and applies WordNet lemmatization.
nltk.download("wordnet", quiet=True)
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
lemmatizer = WordNetLemmatizer()

def extract_concepts(caption):
    nouns = {token.text for token in nlp(str(caption)) if token.pos_ in {"NOUN", "PROPN"}}
    return sorted({lemmatizer.lemmatize(noun.lower()) for noun in nouns})

if not ASSIGNMENTS_PATH.exists() or REBUILD_ASSIGNMENTS:
    records["candidate_concepts_list"] = [
        extract_concepts(caption) for caption in tqdm(records["caption"], desc="Extracting concepts")
    ]
    records["assignable"] = records["candidate_concepts_list"].map(bool)
else:
    print(f"Using existing {ASSIGNMENTS_PATH}; set REBUILD_ASSIGNMENTS=True to recompute.")

In [ ]:
# Encode each distinct concept once, then score image batches on the GPU.
if not ASSIGNMENTS_PATH.exists() or REBUILD_ASSIGNMENTS:
    model, _, preprocess = open_clip.create_model_and_transforms(
        MODEL_NAME,
        pretrained=PRETRAINED,
        precision="fp16",
        device=device,
    )
    model.eval()
    tokenizer = open_clip.get_tokenizer(MODEL_NAME)

    assignable_records = records.loc[records["assignable"]]
    vocabulary = sorted({concept for concepts in assignable_records["candidate_concepts_list"] for concept in concepts})
    text_feature_batches = []
    with torch.inference_mode():
        for start in tqdm(range(0, len(vocabulary), TEXT_BATCH_SIZE), desc="Encoding concepts"):
            tokens = tokenizer(vocabulary[start:start + TEXT_BATCH_SIZE]).to(device)
            features = model.encode_text(tokens)
            features = features / features.norm(dim=-1, keepdim=True)
            text_feature_batches.append(features)
    text_features = torch.cat(text_feature_batches) if text_feature_batches else None
    concept_index = {concept: index for index, concept in enumerate(vocabulary)}

    selected, scores = [], []
    with torch.inference_mode():
        for start in tqdm(range(0, len(assignable_records), IMAGE_BATCH_SIZE), desc="Scoring images"):
            batch = assignable_records.iloc[start:start + IMAGE_BATCH_SIZE]
            pixels = torch.stack([
                preprocess(Image.open(path).convert("RGB")) for path in batch["image_path"]
            ]).to(device=device, dtype=torch.float16)
            image_features = model.encode_image(pixels)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            similarities = 100.0 * image_features @ text_features.T

            for row_index, concepts in enumerate(batch["candidate_concepts_list"]):
                indices = torch.tensor([concept_index[concept] for concept in concepts], device=device)
                probabilities = similarities[row_index, indices].softmax(dim=-1)
                best = int(probabilities.argmax())
                selected.append(concepts[best])
                scores.append(float(probabilities[best].cpu()))

    records["candidate_concepts"] = records["candidate_concepts_list"].map(json.dumps)
    records["selected_concept"] = pd.Series(selected, index=assignable_records.index)
    records["score"] = pd.Series(scores, index=assignable_records.index)
    records[["image_path", "image_id", "caption", "candidate_concepts", "assignable", "selected_concept", "score"]].to_csv(
        ASSIGNMENTS_PATH, index=False
    )

assignments = pd.read_csv(ASSIGNMENTS_PATH)
assignable_count = assignments["selected_concept"].notna().sum()
print(f"Assignable images: {assignable_count}; skipped: {len(assignments) - assignable_count}")
assignments.head()

In [ ]:
# Restrict sampling to sufficiently frequent concepts among the top 100.
concept_counts = assignments["selected_concept"].value_counts()
top_100 = concept_counts.head(100)
eligible = top_100[top_100 >= IMAGES_PER_PAIR]
required_concepts = 2 * NUM_CONCEPT_PAIRS
if len(eligible) < required_concepts:
    raise ValueError(
        f"Need {required_concepts} distinct concepts with at least {IMAGES_PER_PAIR} images; found {len(eligible)}."
    )

top_100.rename("frequency").rename_axis("concept").reset_index().head(100)

In [ ]:
# Seeded sampling supplies the image-level pairing rule not specified by AMP.
rng = np.random.default_rng(RANDOM_SEED)
chosen_concepts = rng.choice(eligible.index.to_numpy(), size=required_concepts, replace=False)
concept_pairs = pd.DataFrame({
    "pair_id": [f"pair_{index:02d}" for index in range(NUM_CONCEPT_PAIRS)],
    "source_concept": chosen_concepts[0::2],
    "target_concept": chosen_concepts[1::2],
})
concept_pairs["source_frequency"] = concept_pairs["source_concept"].map(concept_counts)
concept_pairs["target_frequency"] = concept_pairs["target_concept"].map(concept_counts)
concept_pairs.to_csv(PAIRS_PATH, index=False)
concept_pairs

In [ ]:
# Rebuild the attack-set directories so the manifest and copied files stay in sync.
source_dir = ATTACK_DIR / "source"
target_dir = ATTACK_DIR / "target"
for directory in (source_dir, target_dir):
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True)

manifest_rows = []
for pair in concept_pairs.itertuples(index=False):
    source_pool = assignments[assignments["selected_concept"].eq(pair.source_concept)]
    target_pool = assignments[assignments["selected_concept"].eq(pair.target_concept)]
    source_seed = int(rng.integers(0, 2**32 - 1))
    target_seed = int(rng.integers(0, 2**32 - 1))
    source_rows = source_pool.sample(IMAGES_PER_PAIR, random_state=source_seed).reset_index(drop=True)
    target_rows = target_pool.sample(IMAGES_PER_PAIR, random_state=target_seed).reset_index(drop=True)

    for image_index in range(IMAGES_PER_PAIR):
        sample_id = f"{pair.pair_id}_{image_index:02d}"
        source = source_rows.iloc[image_index]
        target = target_rows.iloc[image_index]
        source_output = source_dir / f"{sample_id}.png"
        target_output = target_dir / f"{sample_id}.png"
        shutil.copy2(source["image_path"], source_output)
        shutil.copy2(target["image_path"], target_output)
        manifest_rows.append({
            "sample_id": sample_id,
            "pair_id": pair.pair_id,
            "source_path": source_output.as_posix(),
            "target_path": target_output.as_posix(),
            "source_concept": pair.source_concept,
            "target_concept": pair.target_concept,
            "source_caption": source["caption"],
            "target_caption": target["caption"],
        })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(ATTACK_DIR / "manifest.csv", index=False)
manifest